# AI Project Builder Assistant

This project builds a **multi-agent AI system** that converts a simple idea into:

- Problem understanding
- System architecture
- Tech stack
- Implementation plan



## Goal
To simulate how a real engineer plans and builds a project using AI agents.



## Features

- Multi-agent workflow
- Tool integration
- State management
- Interactive UI (Gradio)


## Step 1: Setup
We will initialize environment, LLM, and project state.

In [ ]:
!pip install crewai[tools] gradio

# Imports + Secrets

In [ ]:
from crewai import LLM, Agent, Task, Crew
from google.colab import userdata
import gradio as gr

# Load secrets
api_key = userdata.get('OPENAI_API_KEY')
base_url = userdata.get('OPENAI_BASE_URL')

## Initialize LLM

The LLM acts as the **brain** for all agents.

All agents will use this shared model.

In [ ]:
llm = LLM(
    model="gpt-4.1-nano",
    temperature=0.7,
    base_url=base_url,
    api_key=api_key
)

## State Management

We maintain a global state to store:

- User idea
- Analysis
- Architecture
- Tech stack
- Implementation plan

This allows:
- Multi-step reasoning
- UI updates
- Future improvements

In [ ]:
project_state = {
    "idea": None,
    "analysis": None,
    "architecture": None,
    "tech_stack": None,
    "implementation": None
}

## Agent Layer

We now define multiple specialized agents.

Each agent:
- Has a unique role
- Performs a specific task
- Contributes to the overall workflow

---

## Why multiple agents?

Instead of one LLM doing everything, we split responsibilities:

- Better reasoning
- Cleaner outputs
- Scalable design

In [ ]:
idea_analyzer = Agent(
    role="Project Idea Analyzer",
    goal="Understand the user's project idea deeply and break it into problem definition, target users, and key challenges",
    backstory=(
        "You are an expert product thinker who analyzes ideas and extracts core problems, "
        "user needs, and technical challenges before development begins."
    ),
    llm=llm
)

In [ ]:
system_designer = Agent(
    role="System Architecture Designer",
    goal="Design a clear and scalable system architecture based on the analyzed idea",
    backstory=(
        "You are a senior software architect who designs clean, modular, and scalable system architectures. "
        "You focus on components, data flow, and system interactions."
    ),
    llm=llm
)

In [ ]:
tech_advisor = Agent(
    role="Tech Stack Advisor",
    goal="Recommend the most suitable technologies, frameworks, and tools for the project",
    backstory=(
        "You are a technology expert who selects the best tools, frameworks, and models "
        "based on project requirements, scalability, and ease of implementation."
    ),
    llm=llm
)

In [ ]:
planner_agent = Agent(
    role="Implementation Planner",
    goal="Provide a clear, step-by-step plan to build the project from scratch",
    backstory=(
        "You are a project manager and engineer who breaks down projects into actionable steps, "
        "milestones, and development phases."
    ),
    llm=llm
)

## Task Layer (Workflow Design)

We now define tasks for each agent.

Each task:
- Uses a specific agent
- Takes input from previous step
- Produces structured output



## Pipeline Flow

1. Idea Analysis
2. System Design
3. Tech Stack Selection
4. Implementation Planning

Each step updates the project state.

In [ ]:
def run_project_builder(user_idea):
    global project_state

    project_state["idea"] = user_idea

    # --- Step 1: Analysis ---
    analysis_task = Task(
        description=f"""
        Analyze the project idea: '{user_idea}'

        STRICT FORMAT:

        ### Problem
        - point
        - point

        ### Target Users
        - point
        - point

        ### Challenges
        - point
        - point
        """,
        expected_output="Structured bullet points only",
        agent=idea_analyzer
    )

    analysis_result = Crew(tasks=[analysis_task]).kickoff()
    project_state["analysis"] = analysis_result


    # --- Step 2: Architecture ---
    design_task = Task(
        description=f"""
        Based on this analysis:
        {analysis_result}

        STRICT FORMAT:

        ### Components
        - ...

        ### Data Flow
        - ...

        ### Models (if any)
        - ...
        """,
        expected_output="Clean structured output",
        agent=system_designer
    )

    design_result = Crew(tasks=[design_task]).kickoff()
    project_state["architecture"] = design_result


    # --- Step 3: Tech Stack ---
    tech_task = Task(
        description=f"""
        Based on this architecture:
        {design_result}

        STRICT FORMAT:

        ### Backend
        - ...

        ### Frontend
        - ...

        ### ML / AI
        - ...

        ### Tools
        - ...

        ALSO generate project structure using tool
        """,
        expected_output="Structured tech stack",
        agent=tech_advisor
    )

    tech_result = Crew(tasks=[tech_task]).kickoff()
    project_state["tech_stack"] = tech_result


    # --- Step 4: Implementation ---
    plan_task = Task(
        description=f"""
        Based on this tech stack:
        {tech_result}

        STRICT FORMAT:

        ### Phase 1
        - ...

        ### Phase 2
        - ...

        ### Phase 3
        - ...

        ALSO generate starter code using tool
        """,
        expected_output="Step-by-step structured plan",
        agent=planner_agent
    )

    plan_result = Crew(tasks=[plan_task]).kickoff()
    project_state["implementation"] = plan_result

    return project_state

In [ ]:
def format_section(title, content):
    return f"""
<div style="
padding:15px;
border-radius:12px;
background:#0f172a;
margin-bottom:12px;
">

<h3 style="color:#38bdf8; margin-bottom:10px;">{title}</h3>

<div style="
color:#e2e8f0;
font-size:14px;
line-height:1.6;
white-space:pre-wrap;
">
{content}
</div>

</div>
"""

In [ ]:
result = run_project_builder("AI project for smart waste management")

print(result)

In [ ]:
def display_result(result):
    print("\n📌 IDEA:\n", result["idea"])
    print("\n🧠 ANALYSIS:\n", result["analysis"])
    print("\n🏗️ ARCHITECTURE:\n", result["architecture"])
    print("\n🧰 TECH STACK:\n", result["tech_stack"])
    print("\n⚙️ IMPLEMENTATION PLAN:\n", result["implementation"])

In [ ]:
display_result(result)

## Gradio UI

We now build an interactive interface for our AI Project Builder.

Users can:
- Enter a project idea
- Generate full project plan
- View structured output

This transforms our system into a usable application.

In [ ]:
import gradio as gr

## Wrapper Function

We wrap our pipeline function to make it compatible with Gradio.

In [ ]:
def generate_project(idea):
    result = run_project_builder(idea)

    return (
        format_section("🧠 Analysis", result["analysis"]),
        format_section("🏗️ Architecture", result["architecture"]),
        format_section("🧰 Tech Stack", result["tech_stack"]),
        format_section("⚙️ Implementation Plan", result["implementation"])
    )

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🤖 AI Project Builder Assistant
    ### 🚀 Convert your idea into a structured project plan
    ---
    """)

    idea_input = gr.Textbox(
        label="💡 Project Idea",
        placeholder="e.g., AI system for smart waste management",
        lines=2
    )

    generate_btn = gr.Button("🚀 Generate Project Plan", variant="primary")

    gr.Markdown("## 📊 Structured Output")

    with gr.Row():
        with gr.Column():
            analysis_output = gr.HTML()
            architecture_output = gr.HTML()

        with gr.Column():
            tech_output = gr.HTML()
            implementation_output = gr.HTML()

    gr.Examples(
        examples=[
            "AI system for smart waste management",
            "Gesture recognition system for disabled communication",
            "AI-based traffic prediction",
            "Voice AI for emergency response"
        ],
        inputs=idea_input
    )

    generate_btn.click(
        fn=generate_project,
        inputs=idea_input,
        outputs=[
            analysis_output,
            architecture_output,
            tech_output,
            implementation_output
        ],
        show_progress=True
    )

demo.launch()

## Build UI Layout

We create:
- Input textbox
- Output sections
- Button trigger